### Create volume to store checkpoints for streaming to bronze layer

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS maven_test.bronze.checkpoints

### Consumer Script to show the data

In [0]:
from pyspark.sql.functions import from_json, col, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# 1. Retrieve credentials from Azure Key Vault Scope
bootstrap_servers = dbutils.secrets.get(scope="captone_scope", key="kafka-bootstrap-server")
kafka_key         = dbutils.secrets.get(scope="captone_scope", key="kafka-api-key")
kafka_secret      = dbutils.secrets.get(scope="captone_scope", key="kafka-api-secret")

jaas_config = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{kafka_key}" password="{kafka_secret}";'

base_kafka_options = {
    "kafka.bootstrap.servers": bootstrap_servers,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": jaas_config,
    "startingOffsets": "earliest",
    "failOnDataLoss": "false"
}

# ------------------------------------------------------------------------------
# 2. Ingest Stream 1: Online Orders into Bronze
# ------------------------------------------------------------------------------
orders_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("transaction_date", StringType(), True),
    StructField("stock_date", StringType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("channel_mode", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("event_timestamp", StringType(), True)
])

orders_stream = (
    spark.readStream
    .format("kafka")
    .options(**base_kafka_options)
    .option("subscribe", "online_orders")
    .load()
)

df_bronze_orders = (
    orders_stream
    .selectExpr(
        "CAST(key AS STRING) as kafka_partition_key",
        "topic as kafka_topic",
        "partition as kafka_partition",
        "offset as kafka_offset",
        "timestamp as kafka_processing_time",
        "CAST(value AS STRING) as raw_payload"
    )
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", col("kafka_topic"))
)

query_orders = (
    df_bronze_orders.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/maven_test/bronze/checkpoints/kafka_orders_bronze")
    .trigger(availableNow=True)
    .toTable("maven_test.bronze.kafka_online_orders_raw")
)
query_orders.awaitTermination()
print("Bronze online orders table updated.")

# ------------------------------------------------------------------------------
# 3. Ingest Stream 2: Inventory Restock into Bronze
# ------------------------------------------------------------------------------
inventory_schema = StructType([
    StructField("restock_event_id", StringType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("restock_quantity", IntegerType(), True),
    StructField("restock_type", StringType(), True),
    StructField("event_timestamp", StringType(), True)
])

inventory_stream = (
    spark.readStream
    .format("kafka")
    .options(**base_kafka_options)
    .option("subscribe", "inventory_events")
    .load()
)

df_bronze_inventory = (
    inventory_stream
    .selectExpr(
        "CAST(key AS STRING) as kafka_partition_key",
        "topic as kafka_topic",
        "partition as kafka_partition",
        "offset as kafka_offset",
        "timestamp as kafka_processing_time",
        "CAST(value AS STRING) as raw_payload"
    )
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", col("kafka_topic"))
)

query_inventory = (
    df_bronze_inventory.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/maven_test/bronze/checkpoints/kafka_inventory_bronze")
    .trigger(availableNow=True)
    .toTable("maven_test.bronze.kafka_inventory_raw")
)
query_inventory.awaitTermination()
print("Bronze inventory restock table updated.")

In [0]:
%sql
-- 1. Check total rows ingested across both Kafka topics
SELECT 'Online Orders' AS stream_name, COUNT(*) AS bronze_record_count FROM maven_test.bronze.kafka_online_orders_raw
UNION ALL
SELECT 'Inventory Restock', COUNT(*) FROM maven_test.bronze.kafka_inventory_raw;


In [0]:
%sql

-- 2. Confirm that malformed payloads are preserved without schema failure
SELECT kafka_offset, kafka_processing_time, raw_payload, transaction_id, quantity
FROM maven_test.bronze.kafka_online_orders_raw
WHERE transaction_id IS NULL OR quantity <= 0
LIMIT 10;